In [24]:
#數據overview
import pandas as pd
raw_data = pd.read_excel("./updated_data.xlsx")
print("=" * 60)
print("【所有欄位名稱及數據類型】")
print("=" * 60)
for col, dtype in raw_data.dtypes.items():
    print(f"  {col:<40} {dtype}")
print("\n" + "=" * 60)
print("【非數字類型欄位的類別分佈】")
print("=" * 60)
non_numeric_cols = raw_data.select_dtypes(exclude=["number"]).columns
if len(non_numeric_cols) == 0:
    print("  （無非數字類型的欄位）")
else:
    for col in non_numeric_cols:
        print(f"\n▶ 欄位：{col}（類型：{raw_data[col].dtype}）")
        print("-" * 40)
        value_counts = raw_data[col].value_counts(dropna=False)
        for val, count in value_counts.items():
            print(f"    {str(val):<30} 數量：{count}")
        print(f"  → 共 {len(value_counts)} 種類別，{raw_data[col].notna().sum()} 個非空值")

【所有欄位名稱及數據類型】
  Due Date                                 datetime64[ns]
  Ship From                                object
  Ship Via                                 object
  Vessel Name                              object
  Vendor Name                              object
  Incoterm                                 object
  PO Num                                   object
  Cust Num                                 object
  Season                                   object
  Buy No                                   object
  Item                                     object
  RMA Description                          object
  UM                                       object
  Qty                                      float64
  Unit Price With Surcharge                float64
  Total Material Cost (Baht)               float64
  Exwork (Baht)                            int64
  %Exwork (M/L)                            float64
  Freight Cost (Baht)                      int64
  %Freight (O/L)          

In [25]:
column_mapping = {
    'Due Date': 'Date',
    'Ship From': 'Ship_From',
    'Ship Via': 'Ship_Via',
    'Vessel Name': 'Vessel_Name',
    'Vendor Name': 'Vendor_Name',
    'Incoterm': 'Incoterm',
    'PO Num': 'PO_Num',
    'Cust Num': 'Cust_Num',
    'Season': 'Season',
    'Buy No': 'Buy_No',
    'Item': 'Item',
    'Unit Price With Surcharge': 'Unit_Price',
    'Total Material Cost (Baht)': 'Total_Material_Cost',
    'Exwork (Baht)': 'Exwork(M)',
    '%Exwork (M/L)': 'P_Exwork(M)',
    'Freight Cost (Baht)': 'Freight(O)',
    '%Freight (O/L)': 'P_Freight(O)',
    'Local (Baht)': 'Local(Q)',
    '%Local (Q/L)': 'P_Local(Q)',
    'Brokerage (Baht)': 'Brokerage(S)',
    '%Brokerage (S/L)': 'P_Brokerage(S)',
    'Total Import cost (Baht) (M+O+Q+S)': 'Total_Import_cost(U)',
    '% Import Cost (U/L)': 'P_Import_Cost(U)'
}

raw_data = raw_data.rename(columns=column_mapping)
print(raw_data.columns.tolist())

['Date', 'Ship_From', 'Ship_Via', 'Vessel_Name', 'Vendor_Name', 'Incoterm', 'PO_Num', 'Cust_Num', 'Season', 'Buy_No', 'Item', 'RMA Description', 'UM', 'Qty', 'Unit_Price', 'Total_Material_Cost', 'Exwork(M)', 'P_Exwork(M)', 'Freight(O)', 'P_Freight(O)', 'Local(Q)', 'P_Local(Q)', 'Brokerage(S)', 'P_Brokerage(S)', 'Total_Import_cost(U)', 'P_Import_Cost(U)']


In [26]:
features = ['Ship_From', 'Ship_Via', 'Vendor_Name', 'Incoterm', 'Item',
            'Total_Material_Cost', 'Exwork(M)', 'Freight(O)',
            'Local(Q)', 'Brokerage(S)', 'Total_Import_cost(U)', 'Unit_Price']
df = raw_data[features].copy()
#print("size:", df.shape)
#print(df.isnull().sum())
df = df.dropna()
#print("===/n")
#print("size:", df.shape)
#print(df.isnull().sum())

In [27]:
df['Ship_From_Via'] = df['Ship_From'] + '_' + df['Ship_Via']
df['Vendor_From_Via'] = df['Vendor_Name'] + '_' + df['Ship_From_Via']
df['Vendor_Item'] = df['Vendor_Name'] + '_' + df['Item']

#print("Ship_From_Via:")
#print(df['Ship_From_Via'].unique())
#print("\nVendor_From_Via:")
#print(df['Vendor_From_Via'].unique())
#print("\nShip_From_Via size:", df['Ship_From_Via'].nunique())
#print("Vendor_From_Via size:", df['Vendor_From_Via'].nunique())

In [28]:
df['Exwork_is_zero']    = (df['Exwork(M)'] == 0).astype(int)
df['Freight_is_zero']   = (df['Freight(O)'] == 0).astype(int)
df['Local_is_zero']     = (df['Local(Q)'] == 0).astype(int)
df['Brokerage_is_zero'] = (df['Brokerage(S)'] == 0).astype(int)

In [29]:
import numpy as np
num_cols = ['Total_Material_Cost', 'Exwork(M)', 'Freight(O)',
            'Local(Q)', 'Brokerage(S)', 'Total_Import_cost(U)', 'Unit_Price']
df[num_cols] = np.log1p(df[num_cols])
#print(df[num_cols].describe())

In [30]:
cat_cols = ['Vendor_From_Via', 'Incoterm', 'Item']
for col in cat_cols:
    df[col] = df[col].astype('category')

In [31]:
from sklearn.model_selection import train_test_split

X = df[['Vendor_From_Via', 'Incoterm', 'Item', 'Total_Material_Cost', 'Unit_Price',
        'Exwork_is_zero', 'Freight_is_zero', 'Local_is_zero', 'Brokerage_is_zero']]
targets = ['Exwork(M)', 'Freight(O)', 'Local(Q)', 'Brokerage(S)']

X_train, X_test, y_train, y_test = train_test_split(X, df[targets], test_size=0.2, random_state=42)



In [32]:
import xgboost as xgb
models = {}
for target in targets:
    model = xgb.XGBRegressor(enable_categorical=True, random_state=42)
    model.fit(X_train, y_train[target])
    models[target] = model

In [33]:
#incoterm rules //小量提升了
INCOTERM_RULES = {
    'EXW': [],
    'FCA': ['Exwork(M)'],
    'FOB': ['Exwork(M)'],
    'CFR': ['Exwork(M)', 'Freight(O)'],
    'CIF': ['Exwork(M)', 'Freight(O)'],
    'CPT': ['Exwork(M)', 'Freight(O)'],
    'DAP': ['Exwork(M)', 'Freight(O)', 'Local(Q)'],
    'DDP': ['Exwork(M)', 'Freight(O)', 'Local(Q)', 'Brokerage(S)'],
}

predictions = {}
for target in targets:
    predictions[target] = np.expm1(models[target].predict(X_test)).clip(min=0)

# DHL/FED rules //only微量提升
for i, idx in enumerate(X_test.index):
    incoterm = df.loc[idx, 'Incoterm']
    ship_via = str(df.loc[idx, 'Ship_Via']).upper()
    for col in INCOTERM_RULES.get(incoterm, []):
        predictions[col][i] = 0.0
    if any(k in ship_via for k in ['DHL', 'FED']):
        predictions['Exwork(M)'][i] = 0.0


predictions['Total_Import_cost(U)'] = sum(predictions[target] for target in targets)

In [34]:
from sklearn.metrics import mean_absolute_error, r2_score
for target in targets:
    y_true = np.expm1(y_test[target])
    y_pred = predictions[target]

    print(f"\n{target}:")
    print(f"  MAE: {mean_absolute_error(y_true, y_pred):.2f}")
    print(f"  R²:  {r2_score(y_true, y_pred):.4f}")


Exwork(M):
  MAE: 121.67
  R²:  0.7889

Freight(O):
  MAE: 608.13
  R²:  0.9422

Local(Q):
  MAE: 226.28
  R²:  0.8895

Brokerage(S):
  MAE: 358.13
  R²:  0.8838


In [35]:
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score, mean_absolute_error
import numpy as np

X_cv = X
targets_cv = targets
y_cv = df[targets_cv]

kf = KFold(n_splits=5, shuffle=True, random_state=42)

scores = {t: {'r2': [], 'mae': []} for t in targets_cv}

for fold, (train_idx, test_idx) in enumerate(kf.split(X_cv)):
    X_train_fold = X_cv.iloc[train_idx]
    X_test_fold  = X_cv.iloc[test_idx]
    y_train_fold = y_cv.iloc[train_idx]
    y_test_fold  = y_cv.iloc[test_idx]

    models_fold = {}
    for target in targets_cv:
        model = xgb.XGBRegressor(enable_categorical=True, random_state=42)
        model.fit(X_train_fold, y_train_fold[target])
        models_fold[target] = model
    for target in targets_cv:
        y_pred_log = models_fold[target].predict(X_test_fold)
        y_pred = np.expm1(y_pred_log).clip(min=0)
        y_true = np.expm1(y_test_fold[target].values)

        r2 = r2_score(y_true, y_pred)
        mae = mean_absolute_error(y_true, y_pred)
        scores[target]['r2'].append(r2)
        scores[target]['mae'].append(mae)

print("===== 5-Fold Cross-Validation Results =====")
for target in targets_cv:
    r2_mean = np.mean(scores[target]['r2'])
    r2_std  = np.std(scores[target]['r2'])
    mae_mean = np.mean(scores[target]['mae'])
    mae_std  = np.std(scores[target]['mae'])
    print(f"\n{target}:")
    print(f"  R²  : {r2_mean:.4f} (±{r2_std:.4f})")
    print(f"  MAE : {mae_mean:.2f} (±{mae_std:.2f})")

===== 5-Fold Cross-Validation Results =====

Exwork(M):
  R²  : 0.6589 (±0.1729)
  MAE : 164.90 (±45.66)

Freight(O):
  R²  : 0.7113 (±0.3162)
  MAE : 1068.54 (±474.25)

Local(Q):
  R²  : 0.8576 (±0.0320)
  MAE : 264.55 (±30.91)

Brokerage(S):
  R²  : 0.8671 (±0.0185)
  MAE : 393.40 (±30.88)


In [36]:
# model2
df['Vendor_Item'] = df['Vendor_Item'].astype('category')
mask_exw = (df['Incoterm'] == 'EXW') & (df['Exwork_is_zero'] == 0)
df_exw = df[mask_exw].copy()
X_exw = df_exw[['Vendor_Item', 'Total_Material_Cost', 'Vendor_From_Via', 'Item', 'Unit_Price']]
y_exw = df_exw['Exwork(M)']
X_train_exw, X_test_exw, y_train_exw, y_test_exw = train_test_split(
    X_exw, y_exw, test_size=0.2, random_state=42)

model2 = xgb.XGBRegressor(enable_categorical=True, random_state=42)
model2.fit(X_train_exw, y_train_exw)

y_pred_exw = np.expm1(model2.predict(X_test_exw)).clip(min=0)
y_true_exw = np.expm1(y_test_exw)

print(f"Exwork model2 R²:  {r2_score(y_true_exw, y_pred_exw):.4f}")
print(f"Exwork model2 MAE: {mean_absolute_error(y_true_exw, y_pred_exw):.2f}")

Exwork model2 R²:  0.0418
Exwork model2 MAE: 850.60


In [37]:
#model3
import torch
import torch.nn as nn
from sklearn.preprocessing import LabelEncoder

mask_exw = (df['Incoterm'] == 'EXW') & (df['Exwork_is_zero'] == 0)
df_exw = df[mask_exw].copy()

le_vendor_item = LabelEncoder()
le_vendor = LabelEncoder()
le_item = LabelEncoder()

df_exw['Vendor_Item_enc']    = le_vendor_item.fit_transform(df_exw['Vendor_Item'].astype(str))
df_exw['Vendor_From_Via_enc'] = le_vendor.fit_transform(df_exw['Vendor_From_Via'].astype(str))
df_exw['Item_enc']            = le_item.fit_transform(df_exw['Item'].astype(str))

X_exw = df_exw[['Vendor_Item_enc', 'Vendor_From_Via_enc', 'Item_enc', 'Total_Material_Cost']].values
y_exw = df_exw['Exwork(M)'].values

X_train_exw, X_test_exw, y_train_exw, y_test_exw = train_test_split(
    X_exw, y_exw, test_size=0.2, random_state=42)

#轉tensor
X_train_t = torch.FloatTensor(X_train_exw)
y_train_t = torch.FloatTensor(y_train_exw).unsqueeze(1)
X_test_t  = torch.FloatTensor(X_test_exw)

class ExworkNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
    def forward(self, x):
        return self.net(x)

model3 = ExworkNet(X_train_exw.shape[1])
optimizer = torch.optim.Adam(model3.parameters(), lr=0.001)
loss_fn = nn.MSELoss()

for epoch in range(2000):
    model3.train()
    optimizer.zero_grad()
    pred = model3(X_train_t)
    loss = loss_fn(pred, y_train_t)
    loss.backward()
    optimizer.step()


model3.eval()
with torch.no_grad():
    y_pred_log = model3(X_test_t).squeeze().numpy()

y_pred_exw = np.expm1(y_pred_log).clip(min=0)
y_true_exw = np.expm1(y_test_exw)

print(f"神經網絡 Exwork R²:  {r2_score(y_true_exw, y_pred_exw):.4f}")
print(f"神經網絡 Exwork MAE: {mean_absolute_error(y_true_exw, y_pred_exw):.2f}")

神經網絡 Exwork R²:  0.9187
神經網絡 Exwork MAE: 550.61


In [38]:
y_true_total = np.expm1(y_test['Exwork(M)']) + np.expm1(y_test['Freight(O)']) + \
               np.expm1(y_test['Local(Q)']) + np.expm1(y_test['Brokerage(S)'])
y_pred_total = predictions['Total_Import_cost(U)']

print(f"\nTotal_Import_cost(U):")
print(f"  MAE: {mean_absolute_error(y_true_total, y_pred_total):.2f}")
print(f"  R²:  {r2_score(y_true_total, y_pred_total):.4f}")


Total_Import_cost(U):
  MAE: 1180.98
  R²:  0.9404


In [39]:
pred_freight  = predictions['Freight(O)']
pred_local    = predictions['Local(Q)']
pred_brokerage= predictions['Brokerage(S)']
y_true_total = np.expm1(y_test['Exwork(M)']) + np.expm1(y_test['Freight(O)']) + \
               np.expm1(y_test['Local(Q)'])   + np.expm1(y_test['Brokerage(S)'])

#A:全用model1
pred_total_A = predictions['Exwork(M)'] + pred_freight + pred_local + pred_brokerage
print("A：全用model1")
print(f"  MAE: {mean_absolute_error(y_true_total, pred_total_A):.2f}")
print(f"  R²:  {r2_score(y_true_total, pred_total_A):.4f}")

#B：Exwork用 model2 (XGBoost)，其他model1
pred_exw_B = np.zeros(len(X_test))
mask_exw_test = (df.loc[X_test.index, 'Incoterm'] == 'EXW') & \
                (df.loc[X_test.index, 'Exwork_is_zero'] == 0)
X_test_exw_B = df.loc[X_test.index[mask_exw_test], ['Vendor_Item', 'Total_Material_Cost', 'Vendor_From_Via', 'Item', 'Unit_Price']]
pred_exw_B[mask_exw_test.values] = np.expm1(model2.predict(X_test_exw_B)).clip(min=0)

pred_total_B = pred_exw_B + pred_freight + pred_local + pred_brokerage
print("\nB：only Exwork用 model2")
print(f"  MAE: {mean_absolute_error(y_true_total, pred_total_B):.2f}")
print(f"  R²:  {r2_score(y_true_total, pred_total_B):.4f}")

#C：Exwork用model3(神經網絡)，其他model1
pred_exw_C = np.zeros(len(X_test))
df_exw_test_C = df.loc[X_test.index[mask_exw_test]].copy()
df_exw_test_C['Vendor_Item_enc']     = le_vendor_item.transform(df_exw_test_C['Vendor_Item'].astype(str))
df_exw_test_C['Vendor_From_Via_enc'] = le_vendor.transform(df_exw_test_C['Vendor_From_Via'].astype(str))
df_exw_test_C['Item_enc']            = le_item.transform(df_exw_test_C['Item'].astype(str))

X_test_t_C = torch.FloatTensor(
    df_exw_test_C[['Vendor_Item_enc', 'Vendor_From_Via_enc', 'Item_enc', 'Total_Material_Cost']].values)
model3.eval()
with torch.no_grad():
    pred_exw_C[mask_exw_test.values] = np.expm1(model3(X_test_t_C).squeeze().numpy()).clip(min=0)

pred_total_C = pred_exw_C + pred_freight + pred_local + pred_brokerage
print("\nC：only Exwork用model3")
print(f"  MAE: {mean_absolute_error(y_true_total, pred_total_C):.2f}")
print(f"  R²:  {r2_score(y_true_total, pred_total_C):.4f}")

A：全用model1
  MAE: 1180.98
  R²:  0.9404

B：only Exwork用 model2
  MAE: 1111.47
  R²:  0.9422

C：only Exwork用model3
  MAE: 1182.83
  R²:  0.9404


In [40]:
# predict_import_cost function的限制
AIR_SHIP_FROMS = [
    'HK BY AIR', 'TAIWAN KEELUNG BY AIR', 'JAPAN OSAKA BY AIR',
    'VIETNAM BY AIR', 'CHINA SHANGHAI BY AIR', 'JAPAN OSAKA TO MM BY AIR',
    'ITALY BY AIR', 'KOREA BUSAN BY AIR', 'TAIWAN KEELUNG TO MM BY AIR',
    'HK TO MM BY AIR', 'FRANCE BY AIR', 'FRANCE TO MM BY AIR',
    'CHINA GUANGZHOU BY AIR', 'KOREA BUSAN TO MM BY AIR', 'CHINA XIAMEN BY AIR',
    'CHINA SHANGHAI TO MM BY AIR', 'VIETNAM TO MM BY AIR', 'NEW ZELAND BY AIR',
    'TAIWAN KAOHSIUNG TO MM BY AIR', 'THAILAND TO MM BY AIR',
    'TAIWAN KAOHSIUNG BY AIR', 'CHINA SHENZHEN BY AIR'
]

INVALID_INCOTERM_VIA = [
    ('FOB', 'DHL'), ('FOB', 'FED'),
    ('CIF', 'FED'),
    ('DAP', 'DHL'), ('DAP', 'FED'),
]

def is_valid_combination(ship_from, ship_via, incoterm):
    ship_via_upper = ship_via.upper()
    # 規則1：BY AIR的ship_from只能配AIR/DHL/FED
    if ship_from in AIR_SHIP_FROMS:
        if not any(k in ship_via_upper for k in ['AIR', 'DHL', 'FED']):
            return False, f"'{ship_from}' 只能配 AIR/DHL/FED，不能配 '{ship_via}'"
    # 規則2：非法組合of Incoterm + Ship_Via
    for inv_inc, inv_via in INVALID_INCOTERM_VIA:
        if incoterm == inv_inc and inv_via in ship_via_upper:
            return False, f"'{incoterm}' 不能與 '{ship_via}' 搭配"
    # 規則345678。。。？

    return True, None

In [41]:
#輸入：vendor_name, ship_from, ship_via, item, total_material_cost, incoterm
#輸出：predict的Total_Import_cost
def predict_import_cost(vendor_name, ship_from, ship_via, item, total_material_cost, unit_price, incoterm, use_model='B'):



    valid, reason = is_valid_combination(ship_from, ship_via, incoterm)
    if not valid:
        print(f"Warning: Unreasonable combination in input! Reason：{reason}")
        choice = input("Enter 'y' to ignore the warning and continue, any other key to exit:").strip().lower()
        if choice != 'y':
            print("Exited.")
            return None



    ship_from_via   = f"{ship_from}_{ship_via}"
    vendor_from_via = f"{vendor_name}_{ship_from_via}"
    vendor_item     = f"{vendor_name}_{item}"

    zero_cols = INCOTERM_RULES.get(incoterm, [])
    exwork_is_zero   = 1 if 'Exwork(M)'    in zero_cols else 0
    freight_is_zero  = 1 if 'Freight(O)'   in zero_cols else 0
    local_is_zero    = 1 if 'Local(Q)'     in zero_cols else 0
    brokerage_is_zero= 1 if 'Brokerage(S)' in zero_cols else 0

    input_m1 = pd.DataFrame({
        'Vendor_From_Via':  [vendor_from_via],
        'Incoterm':         [incoterm],
        'Item':             [item],
        'Total_Material_Cost': [np.log1p(total_material_cost)],
        'Unit_Price':       [np.log1p(unit_price)],
        'Exwork_is_zero':   [exwork_is_zero],
        'Freight_is_zero':  [freight_is_zero],
        'Local_is_zero':    [local_is_zero],
        'Brokerage_is_zero':[brokerage_is_zero],
    })
    for col in ['Vendor_From_Via', 'Incoterm', 'Item']:
        input_m1[col] = input_m1[col].astype('category')

    # 預測 Freight, Local, Brokerage（用model1）
    results = {}
    for target in ['Freight(O)', 'Local(Q)', 'Brokerage(S)']:
        results[target] = np.expm1(models[target].predict(input_m1))[0].clip(min=0)

    # 預測 Exwork（用2，3）
    if exwork_is_zero == 1:
        results['Exwork(M)'] = 0.0
    elif use_model == 'A':
        results['Exwork(M)'] = np.expm1(models['Exwork(M)'].predict(input_m1))[0].clip(min=0)
    elif use_model == 'B':
        input_m2 = pd.DataFrame({
            'Vendor_Item':        [vendor_item],
            'Total_Material_Cost':[np.log1p(total_material_cost)],
            'Vendor_From_Via':    [vendor_from_via],
            'Item':               [item],
            'Unit_Price':         [np.log1p(unit_price)]
        })
        for col in ['Vendor_Item', 'Vendor_From_Via', 'Item']:
            input_m2[col] = input_m2[col].astype('category')
        results['Exwork(M)'] = np.expm1(model2.predict(input_m2))[0].clip(min=0)
    elif use_model == 'C':
        enc_vi  = le_vendor_item.transform([vendor_item])[0]
        enc_vfv = le_vendor.transform([vendor_from_via])[0]
        enc_i   = le_item.transform([item])[0]
        x_t = torch.FloatTensor([[enc_vi, enc_vfv, enc_i, np.log1p(total_material_cost)]])
        model3.eval()
        with torch.no_grad():
            results['Exwork(M)'] = np.expm1(model3(x_t).item()).clip(min=0)


    for col in zero_cols:
        results[col] = 0.0

    results['Total_Import_cost(U)'] = sum(results[target] for target in targets)

    for key, val in results.items():
        print(f"{key}: {val:.2f} Baht")
    return results



In [42]:
predict_import_cost(
    vendor_name='KINGWHALE CORPORATION',
    ship_from='TAIWAN KEELUNG TO MM',
    ship_via='SEA',
    item='CKN',
    total_material_cost=776889,
    unit_price=10000,
    incoterm='FOB',
    use_model='B'
)

Freight(O): 20773.44 Baht
Local(Q): 1825.19 Baht
Brokerage(S): 7787.17 Baht
Exwork(M): 0.00 Baht
Total_Import_cost(U): 30385.79 Baht


{'Freight(O)': 20773.44140625,
 'Local(Q)': 1825.18701171875,
 'Brokerage(S)': 7787.1650390625,
 'Exwork(M)': 0.0,
 'Total_Import_cost(U)': 30385.79345703125}

In [43]:
def find_best_combinations(
    item,                          # 必填
    total_material_cost=50000,     # 選填，預設50000
    unit_price=100.0,
    vendor_options=None,           # 以下都是選填
    ship_from_options=None,
    ship_via_options=None,
    incoterm_options=None,
    use_model='B',
    top_n=5
):
    all_vendors    = df['Vendor_Name'].unique().tolist()
    all_ship_from  = df['Ship_From'].unique().tolist()
    all_ship_via   = df['Ship_Via'].unique().tolist()
    all_incoterms  = list(INCOTERM_RULES.keys())

    vendors   = vendor_options      if vendor_options      else all_vendors
    ship_froms= ship_from_options   if ship_from_options   else all_ship_from
    ship_vias = ship_via_options    if ship_via_options    else all_ship_via
    incoterms = incoterm_options    if incoterm_options    else all_incoterms


    if ship_from_options and ship_via_options and incoterm_options:
        all_invalid = all(
            not is_valid_combination(sf, sv, inc)[0]
            for sf in ship_froms
            for sv in ship_vias
            for inc in incoterms
        )
        if all_invalid:
            print("Warning: All combinations you entered are invalid. No feasible combination exists.")
            choice = input("Enter 'y' to ignore warning and continue, any other key to exit: ").strip().lower()
            if choice != 'y':
                print("Exiting program.")
                return
            else:
                print("Continuing...\n")

    results_list = []
    skipped_invalid = 0
    skipped_error = 0

    for v in vendors:
        for sf in ship_froms:
            for sv in ship_vias:
                for inc in incoterms:
                    valid, _ = is_valid_combination(sf, sv, inc)
                    if not valid:
                        skipped_invalid += 1
                        continue
                    try:
                        r = predict_import_cost(
                            vendor_name=v,
                            ship_from=sf,
                            ship_via=sv,
                            item=item,
                            total_material_cost=total_material_cost,
                            unit_price=unit_price,
                            incoterm=inc,
                            use_model=use_model
                        )
                        if r is None:
                            skipped_error += 1
                            continue
                        results_list.append({
                            'Vendor_Name': v,
                            'Ship_From':   sf,
                            'Ship_Via':    sv,
                            'Incoterm':    inc,
                            'Exwork(M)':   round(r['Exwork(M)'], 2),
                            'Freight(O)':  round(r['Freight(O)'], 2),
                            'Local(Q)':    round(r['Local(Q)'], 2),
                            'Brokerage(S)':round(r['Brokerage(S)'], 2),
                            'Total_Import_cost(U)': round(r['Total_Import_cost(U)'], 2),
                        })
                    except:
                        skipped_error += 1

    print(f"Valid combinations: {len(results_list)}")
    print(f"Skipped due to restrictions: {skipped_invalid}")
    print(f"Skipped due to model errors: {skipped_error}")

    if not results_list:
        print("No valid combinations found.")
        return

    actual_top_n = min(top_n, len(results_list))
    if actual_top_n < top_n:
        print(f"Only {actual_top_n} valid combination(s) found, fewer than requested {top_n}.")

    results_df = pd.DataFrame(results_list)
    top_results = results_df.nsmallest(actual_top_n, 'Total_Import_cost(U)').reset_index(drop=True)

    print(f"\n=== Top {actual_top_n} combinations with smallest Total_Import_cost ===")
    print(f"Item: {item}, Total_Material_Cost: {total_material_cost} Baht\n")
    for i, row in top_results.iterrows():
        print(f"--- Option {i+1} ---")
        print(f"  Vendor:    {row['Vendor_Name']}")
        print(f"  Ship_From: {row['Ship_From']}")
        print(f"  Ship_Via:  {row['Ship_Via']}")
        print(f"  Incoterm:  {row['Incoterm']}")
        print(f"  Exwork(M):         {row['Exwork(M)']:>10.2f} Baht")
        print(f"  Freight(O):        {row['Freight(O)']:>10.2f} Baht")
        print(f"  Local(Q):          {row['Local(Q)']:>10.2f} Baht")
        print(f"  Brokerage(S):      {row['Brokerage(S)']:>10.2f} Baht")
        print(f"  Total_Import_cost: {row['Total_Import_cost(U)']:>10.2f} Baht\n")

    return top_results

In [44]:
#function testing
find_best_combinations(
    item='Knitted fabric',
    total_material_cost=776889,
    incoterm_options=['FOB', 'EXW', 'CIF'],
    vendor_options=['KINGWHALE CORPORATION', 'AVERY DENNISON HONG  KONG   B.V.'],
    ship_from_options=['HK', 'TAIWAN KEELUNG TO MM'],
    ship_via_options=['SEA', 'DHL'],
)

Valid combinations: 0
Skipped due to restrictions: 4
Skipped due to model errors: 20
No valid combinations found.


In [45]:
import joblib
import json

# 保存模型
joblib.dump(models, 'models.pkl')
joblib.dump(model2, 'model2.pkl')

# 保存下拉列表数据
vendor_list = df['Vendor_Name'].unique().tolist()
ship_from_list = df['Ship_From'].unique().tolist()
ship_via_list = df['Ship_Via'].unique().tolist()
item_list = df['Item'].unique().tolist()
incoterm_list = list(INCOTERM_RULES.keys())

with open('dropdown_data.json', 'w', encoding='utf-8') as f:
    json.dump({
        'vendors': vendor_list,
        'ship_froms': ship_from_list,
        'ship_vias': ship_via_list,
        'items': item_list,
        'incoterms': incoterm_list
    }, f, ensure_ascii=False)

print("✅ 模型和数据保存成功！")
print(f"Vendors: {len(vendor_list)}, Ship From: {len(ship_from_list)}, Items: {len(item_list)}")

✅ 模型和数据保存成功！
Vendors: 60, Ship From: 31, Items: 41


In [46]:
# 生成完整版 app.py（包含 Unit_Price）
complete_app_code = '''
import numpy as np
import pandas as pd
import joblib
import json
from flask import Flask, render_template, request, jsonify
from flask_cors import CORS

app = Flask(__name__)
CORS(app)

# Load models
models = joblib.load('models.pkl')
model2 = joblib.load('model2.pkl')

# Load dropdown data
with open('dropdown_data.json', 'r', encoding='utf-8') as f:
    dropdown_data = json.load(f)

VENDOR_LIST = dropdown_data['vendors']
SHIP_FROM_LIST = dropdown_data['ship_froms']
SHIP_VIA_LIST = dropdown_data['ship_vias']
ITEM_LIST = dropdown_data['items']
INCOTERM_LIST = dropdown_data['incoterms']

# ========== 规则 3: Incoterm 费用强制归零规则 ==========
INCOTERM_RULES = {
    'EXW': ['Freight(O)', 'Local(Q)', 'Brokerage(S)'],
    'FCA': ['Freight(O)', 'Local(Q)', 'Brokerage(S)'],
    'FOB': ['Freight(O)', 'Local(Q)', 'Brokerage(S)'],
    'CFR': ['Local(Q)', 'Brokerage(S)'],
    'CIF': ['Local(Q)', 'Brokerage(S)'],
    'CPT': ['Local(Q)', 'Brokerage(S)'],
    'DAP': ['Brokerage(S)'],
    'DDP': [],
}

# ========== 规则 1: BY AIR 发货地限制 ==========
AIR_SHIP_FROMS = [
    'HK BY AIR', 'TAIWAN KEELUNG BY AIR', 'JAPAN OSAKA BY AIR',
    'VIETNAM BY AIR', 'CHINA SHANGHAI BY AIR', 'JAPAN OSAKA TO MM BY AIR',
    'ITALY BY AIR', 'KOREA BUSAN BY AIR', 'TAIWAN KEELUNG TO MM BY AIR',
    'HK TO MM BY AIR', 'FRANCE BY AIR', 'FRANCE TO MM BY AIR',
    'CHINA GUANGZHOU BY AIR', 'KOREA BUSAN TO MM BY AIR', 'CHINA XIAMEN BY AIR',
    'CHINA SHANGHAI TO MM BY AIR', 'VIETNAM TO MM BY AIR', 'NEW ZELAND BY AIR',
    'TAIWAN KAOHSIUNG TO MM BY AIR', 'THAILAND TO MM BY AIR',
    'TAIWAN KAOHSIUNG BY AIR', 'CHINA SHENZHEN BY AIR'
]

# ========== 规则 2: Incoterm + 快递方式限制 ==========
INVALID_INCOTERM_VIA = [
    ('FOB', 'DHL'), ('FOB', 'FED'),
    ('CIF', 'FED'),
    ('DAP', 'DHL'), ('DAP', 'FED'),
]

def is_valid_combination(ship_from, ship_via, incoterm):
    ship_via_upper = ship_via.upper()
    if ship_from in AIR_SHIP_FROMS:
        if not any(k in ship_via_upper for k in ['AIR', 'DHL', 'FED']):
            return False, f"'{ship_from}' can only use AIR/DHL/FED, not '{ship_via}'"
    for inv_inc, inv_via in INVALID_INCOTERM_VIA:
        if incoterm == inv_inc and inv_via in ship_via_upper:
            return False, f"'{incoterm}' cannot be used with '{ship_via}'"
    return True, None

def predict_cost(vendor_name, ship_from, ship_via, item, total_material_cost, unit_price, incoterm):
    ship_from_via = f"{ship_from}_{ship_via}"
    vendor_from_via = f"{vendor_name}_{ship_from_via}"
    vendor_item = f"{vendor_name}_{item}"
    
    zero_cols = INCOTERM_RULES.get(incoterm, [])
    exwork_is_zero = 1 if 'Exwork(M)' in zero_cols else 0
    freight_is_zero = 1 if 'Freight(O)' in zero_cols else 0
    local_is_zero = 1 if 'Local(Q)' in zero_cols else 0
    brokerage_is_zero = 1 if 'Brokerage(S)' in zero_cols else 0
    
    input_data = pd.DataFrame({
        'Vendor_From_Via': [vendor_from_via],
        'Incoterm': [incoterm],
        'Item': [item],
        'Total_Material_Cost': [np.log1p(max(total_material_cost, 0.01))],
        'Unit_Price': [np.log1p(max(unit_price, 0.01))],
        'Exwork_is_zero': [exwork_is_zero],
        'Freight_is_zero': [freight_is_zero],
        'Local_is_zero': [local_is_zero],
        'Brokerage_is_zero': [brokerage_is_zero],
    })
    
    for col in ['Vendor_From_Via', 'Incoterm', 'Item']:
        input_data[col] = input_data[col].astype('category')
    
    results = {}
    for target in ['Exwork(M)', 'Freight(O)', 'Local(Q)', 'Brokerage(S)']:
        pred = models[target].predict(input_data)[0]
        results[target] = max(0, float(np.expm1(pred)))
    
    # 规则 4: DHL/FED 时 Exwork 强制为 0
    if any(k in ship_via.upper() for k in ['DHL', 'FED']):
        results['Exwork(M)'] = 0.0
    
    # 规则 3: 根据 Incoterm 强制归零
    for col in zero_cols:
        results[col] = 0.0
    
    results['Total_Import_cost(U)'] = sum(results.values())
    
    return results

@app.route('/')
def index():
    return render_template('index.html', 
                         vendors=VENDOR_LIST,
                         ship_froms=SHIP_FROM_LIST,
                         ship_vias=SHIP_VIA_LIST,
                         incoterms=INCOTERM_LIST,
                         items=ITEM_LIST)

@app.route('/predict', methods=['POST'])
def predict():
    try:
        data = request.get_json()
        
        vendor = data.get('vendor')
        ship_from = data.get('ship_from')
        ship_via = data.get('ship_via')
        item = data.get('item')
        material_cost = float(data.get('material_cost', 50000))
        unit_price = float(data.get('unit_price', 100))
        incoterm = data.get('incoterm')
        
        valid, reason = is_valid_combination(ship_from, ship_via, incoterm)
        if not valid:
            return jsonify({'error': reason}), 400
        
        result = predict_cost(vendor, ship_from, ship_via, item, material_cost, unit_price, incoterm)
        
        return jsonify({
            'success': True,
            'exwork': round(result['Exwork(M)'], 2),
            'freight': round(result['Freight(O)'], 2),
            'local': round(result['Local(Q)'], 2),
            'brokerage': round(result['Brokerage(S)'], 2),
            'total': round(result['Total_Import_cost(U)'], 2)
        })
    except Exception as e:
        import traceback
        traceback.print_exc()
        return jsonify({'error': str(e)}), 500

@app.route('/optimize', methods=['POST'])
def optimize():
    try:
        data = request.get_json()
        
        item = data.get('item')
        material_cost = float(data.get('material_cost', 50000))
        unit_price = float(data.get('unit_price', 100))
        
        vendors = data.get('vendors', [])
        ship_froms = data.get('ship_froms', [])
        ship_vias = data.get('ship_vias', [])
        incoterms = data.get('incoterms', [])
        top_n = data.get('top_n', 5)
        
        if not vendors:
            vendors = VENDOR_LIST
        if not ship_froms:
            ship_froms = SHIP_FROM_LIST
        if not ship_vias:
            ship_vias = SHIP_VIA_LIST
        if not incoterms:
            incoterms = INCOTERM_LIST
        
        print(f"Optimizing: item={item}, material_cost={material_cost}, unit_price={unit_price}")
        
        results = []
        for vendor in vendors:
            for ship_from in ship_froms:
                for ship_via in ship_vias:
                    for incoterm in incoterms:
                        valid, _ = is_valid_combination(ship_from, ship_via, incoterm)
                        if not valid:
                            continue
                        
                        try:
                            result = predict_cost(vendor, ship_from, ship_via, item, material_cost, unit_price, incoterm)
                            results.append({
                                'vendor': vendor,
                                'ship_from': ship_from,
                                'ship_via': ship_via,
                                'incoterm': incoterm,
                                'exwork': round(result['Exwork(M)'], 2),
                                'freight': round(result['Freight(O)'], 2),
                                'local': round(result['Local(Q)'], 2),
                                'brokerage': round(result['Brokerage(S)'], 2),
                                'total': round(result['Total_Import_cost(U)'], 2)
                            })
                        except Exception:
                            continue
        
        results.sort(key=lambda x: x['total'])
        top_results = results[:top_n]
        
        print(f"Found {len(results)} valid combinations")
        
        return jsonify({
            'success': True,
            'combinations': top_results,
            'total_searched': len(results)
        })
        
    except Exception as e:
        import traceback
        traceback.print_exc()
        return jsonify({'error': str(e)}), 500

if __name__ == '__main__':
    app.run(debug=True, host='0.0.0.0', port=5000)
'''

# Save as app.py
with open('app.py', 'w', encoding='utf-8') as f:
    f.write(complete_app_code)

print("✅ app.py 已生成（包含 Unit_Price）")

✅ app.py 已生成（包含 Unit_Price）
